# 04 — Transformer fine-tuning (Task D)

**Project:** Irony detection in tweets — TweetEval `irony` (SemEval-2018 Task 3)
**Goal of this notebook:** fine-tune a pre-trained transformer (BERT family) to classify tweets as **ironic (1)** or **not ironic (0)**, and compare it with the classical models (TF-IDF + NB / LogReg / SVM / RF / KNN) of the other notebooks.
**Author:** Mohamed Chaker Iben Hadj Amor
**Tools:** Python 3.12 · PyTorch (CUDA, RTX 5060 Ti) · Hugging Face Transformers & Datasets · pandas · scikit-learn

**Why a transformer?** TF-IDF models only count words: they lose word order and context. Irony lives precisely in the *contrast* between the words and the situation ("I *love* waiting 2 hours at the bank"). A transformer is pre-trained on billions of sentences and reads each word **in context**, so it should capture this contrast better.

**Plan**
1. Setup and GPU check
2. Loading the data
3. Data quality: corrupted rows
4. Hugging Face dataset format
5. Tokenization
6. Model, metrics and training (DistilBERT)
7. Results so far and next steps

## 1. Setup and GPU check

The first cell lets the notebook find the shared `src/` folder (one level up). The imports from `src/` are commented out for now: those files are written by other team members and are still empty.

Fine-tuning a transformer is heavy computation, so we first make sure **PyTorch can use the NVIDIA GPU**. Checking `torch.cuda.is_available()` is not enough: we also multiply two random 1000×1000 matrices **on the GPU**, which proves the GPU really computes.

**Sanity check of the result:** each entry of `x @ x` is a sum of 1000 products of two random numbers in [0, 1). Their average is 0.5 × 0.5 = 0.25, so each entry ≈ 250, and the sum of the 1,000,000 entries ≈ **250,000,000** — which is what we get.

In [1]:
import sys
sys.path.append("..")

#from src.config import *
#from src.data import load_split
#from src.cleaning import clean_text
#from src.evaluate import evaluate, save_results

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

x = torch.rand(1000, 1000, device="cuda")
y = x @ x
print("Result:", y.sum().item())

PyTorch version: 2.15.0+cu132
CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Ti
Result: 250409344.0


## 2. Loading the data

The official TweetEval files store **text and labels in separate files**, one tweet per line: line *N* of `train_text.txt` goes with line *N* of `train_labels.txt`.

Choices made in `load_raw_split()`:
- **Read line by line, not with `pd.read_csv`**: hundreds of tweets contain commas and quotes, which `read_csv` would interpret as column separators and break the tweets apart.
- **`encoding="utf-8"`**: tweets contain emojis and special characters.
- **`.splitlines()`**: splits on line breaks without creating an empty extra row at the end of the file.
- **`assert`**: if a text file and its labels file ever had different lengths, every tweet after the mismatch would silently get the wrong label. The assert turns this into a clear error.

We check that we get the official sizes: **2862 / 955 / 784** tweets.

In [3]:
import pandas as pd

DATA_DIR = "../data/raw"

def load_raw_split(split):
    """Read one split ("train", "val" or "test") of TweetEval irony into a DataFrame."""
    with open(f"{DATA_DIR}/{split}_text.txt", encoding="utf-8") as f:
        texts = f.read().splitlines()

    with open(f"{DATA_DIR}/{split}_labels.txt", encoding="utf-8") as f:
        labels = [int(line) for line in f.read().splitlines()]

    assert len(texts) == len(labels), f"{split}: {len(texts)} texts but {len(labels)} labels"

    return pd.DataFrame({"text": texts, "label": labels})


train_df = load_raw_split("train")
val_df = load_raw_split("val")
test_df = load_raw_split("test")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (2862, 2)
val:   (955, 2)
test:  (784, 2)


In [4]:
train_df.head()

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1


## 3. Data quality: corrupted rows in the training set

Looking at the data before training, we noticed user mentions are already anonymized as `@user` and hashtags are kept.

While measuring tweet lengths, we found that the longest training "tweet" was in fact **about ten tweets merged into one line**, containing pieces of the original competition file: tweet IDs and **their own labels**, separated by tab characters (`\t`). Such a row has a single label while containing both ironic and non-ironic tweets, so **its label is unreliable**.

Searching all splits for tab characters:

| Split | Corrupted rows |
|---|---|
| train | **3** (1000, 329 and 249 characters long) |
| validation | 0 |
| test | 0 |

We **remove these 3 rows from the training set only** (2862 → **2859**). This is 0.1% of the data, so it doesn't change the class balance. The test set is clean and is never modified.

`reset_index(drop=True)` renumbers the rows after the removal; without it, the conversion to the Hugging Face format would add a useless extra index column.

In [5]:
corrupted = train_df["text"].str.contains("\t")

print("Corrupted rows found:", corrupted.sum())
print(train_df.loc[corrupted, "text"].str.len())

train_df = train_df[~corrupted].reset_index(drop=True)
print("train after removal:", train_df.shape)

Corrupted rows found: 3
1295    1000
1753     329
2773     249
Name: text, dtype: int64
train after removal: (2859, 2)


## 4. Hugging Face dataset format

The tokenizer and the `Trainer` work on Hugging Face `Dataset` objects, not on pandas DataFrames. We convert each split and group the three of them in a `DatasetDict`, so that later **one** `.map()` call can tokenize all splits at once. The split is named `validation`, following the Hugging Face convention.

The first training example already shows **verbal irony**: *"really excited for the next 3 weeks"* — said by someone on crutches. Positive words, negative situation, label 1.

In [6]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2859
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 955
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 784
    })
})
{'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  ', 'label': 1}


## 5. Tokenization

A neural network only works with numbers. The **tokenizer** converts each tweet into a list of **token IDs**:

`"really excited for my crutches"` → `["really", "excited", "for", "my", "crut", "##ches"]` → `[2428, 7568, ...]`

Key points:
- Tokens are often **pieces of words** (subwords): rare words are split into known pieces, so the model never meets a completely unknown word.
- **Each model has its own tokenizer**, and we must use the one it was pre-trained with — otherwise the IDs mean nothing to the model.
- The tokenizer adds **special tokens** (`[CLS]` at the start, `[SEP]` at the end for BERT models).
- The **attention mask** marks real tokens (1) versus padding (0).

We start with **DistilBERT** (`distilbert-base-uncased`), a smaller, faster version of BERT, to build and debug the full pipeline.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = ds["train"][0]["text"]
encoded = tokenizer(example)

print("Tweet:    ", example)
print("input_ids:", encoded["input_ids"])
print("tokens:   ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("attention:", encoded["attention_mask"])

Tweet:     seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  
input_ids: [101, 3773, 4903, 2140, 3788, 1059, 1013, 13675, 4904, 8376, 3084, 2033, 2428, 7568, 2005, 1996, 2279, 1017, 3134, 1997, 2026, 2166, 102]
tokens:    ['[CLS]', 'seeing', 'pp', '##l', 'walking', 'w', '/', 'cr', '##ut', '##ches', 'makes', 'me', 'really', 'excited', 'for', 'the', 'next', '3', 'weeks', 'of', 'my', 'life', '[SEP]']
attention: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### 5.1 Does the tokenizer keep the irony signals?

We tokenize a sarcastic sentence with two tokenizers:

| Tokenizer | Result for `"Oh GREAT, another Monday 🙂"` |
|---|---|
| DistilBERT (uncased) | `oh`, `great`, …, **`[UNK]`** |
| RoBERTa | `Oh`, `ĠGREAT`, …, **`ĠðŁĻĤ`** |

- DistilBERT-**uncased** lowercases everything: **"GREAT" (shouting) becomes "great"** (neutral).
- It doesn't know the emoji and replaces it with **`[UNK]`** (unknown): the 🙂 is lost.
- RoBERTa **keeps the capitals**, and encodes the emoji as a real token (the strange characters are the emoji's bytes; `Ġ` marks a preceding space).

**Conclusion:** DistilBERT-uncased loses two irony signals. This is why our final model will come from the **RoBERTa family**; DistilBERT is kept as a baseline for comparison.

In [8]:
roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

test_tweet = "Oh GREAT, another Monday 🙂"

for name, tok in [("DistilBERT", tokenizer), ("RoBERTa", roberta_tokenizer)]:
    ids = tok(test_tweet)["input_ids"]
    print(f"{name:10} -> {tok.convert_ids_to_tokens(ids)}")

DistilBERT -> ['[CLS]', 'oh', 'great', ',', 'another', 'monday', '[UNK]', '[SEP]']
RoBERTa    -> ['<s>', 'Oh', 'ĠGREAT', ',', 'Ġanother', 'ĠMonday', 'ĠðŁĻĤ', '</s>']


### 5.2 Choosing the maximum length

Every model has a maximum input length, and we need to choose `max_length`: too short cuts tweets, too long wastes memory and time. We **measure** the length in tokens of every training tweet instead of guessing:

| Statistic | Value |
|---|---|
| mean | ≈ 23 tokens |
| median | 22 |
| 99th percentile | 44 |
| max (after removing the corrupted rows) | 60 |

Before the corrupted rows were removed, the max was 294 — that outlier is how we found them. Note that the 99th percentile was 44 in both cases: **percentiles are robust to outliers, the max is not**.

We choose **`MAX_LENGTH = 64`**: it covers 100% of the training tweets and leaves some margin for other tokenizers. Also note that **characters ≠ tokens**: a short tweet containing shortened links (`t.co/Q2WB7riAvK`) produces many tokens, because random strings are split into many small pieces.

In [9]:
token_lengths = [len(tokenizer(text)["input_ids"]) for text in train_df["text"]]

lengths = pd.Series(token_lengths)
print(lengths.describe())
print("99th percentile:", lengths.quantile(0.99))

count    2859.000000
mean       22.730325
std         9.082953
min         3.000000
25%        15.000000
50%        22.000000
75%        30.000000
max        60.000000
dtype: float64
99th percentile: 44.0


In [10]:
print(train_df.loc[lengths.idxmax(), "text"])

@user Hey #Russia ! MAKE MONEY #TWEETING YET?||http://t.co/Q2WB7riAvK <-This #FOXNews Clip explains how||https://t.co/cpJZkEgNe7 


### 5.3 Tokenizing the whole dataset

`.map()` applies the tokenizer to every tweet of every split. With `batched=True`, tweets are processed in chunks, which is much faster. Tweets longer than 64 tokens are truncated.

We **don't pad here**. Padding is done later, batch by batch (*dynamic padding*): each batch is only padded to the length of its own longest tweet (often ~30 tokens) instead of always 64.

The `token_type_ids` column is only useful for tasks with two sentences (e.g. question + answer); it is ignored here and dropped automatically by the `Trainer`.

In [11]:
MAX_LENGTH = 64

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_ds = ds.map(tokenize, batched=True)

print(tokenized_ds)
print(tokenized_ds["train"][0])

Map:   0%|          | 0/2859 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2859
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 955
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 784
    })
})
{'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  ', 'label': 1, 'input_ids': [101, 3773, 4903, 2140, 3788, 1059, 1013, 13675, 4904, 8376, 3084, 2033, 2428, 7568, 2005, 1996, 2279, 1017, 3134, 1997, 2026, 2166, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 6. Model, metrics and training

### 6.1 The model: pre-trained body + new classification head

`AutoModelForSequenceClassification` loads DistilBERT's **pre-trained body** (which already "knows" English) and adds a **new classification head** with **2 outputs** (one score per class).

The warning when loading is expected:
- **UNEXPECTED**: the weights of the head used during pre-training (guessing hidden words) — not needed for our task, so they are dropped.
- **MISSING**: our new head (`pre_classifier`, `classifier`), which starts **random** and is what fine-tuning will train.

The model has about **67 million parameters**; less than 1% of them belong to the new head. Fine-tuning reuses the pre-trained knowledge and adapts it to irony.

In [12]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "non_irony", 1: "irony"}
label2id = {"non_irony": 0, "irony": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(model.classifier)
print(f"Parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)
Parameters: 66,955,010


### 6.2 Dynamic padding

The GPU processes tweets in batches, and a batch must be a rectangle: every tweet the same length. The **data collator** pads each batch to the length of its longest tweet with a padding token, and sets the attention mask to 0 on the padding so the model ignores it.

In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### 6.3 Evaluation metric

After each epoch, the model is scored on the **validation** set (never the test set). The model outputs two scores (*logits*) per tweet; `argmax` picks the class with the highest score.

We report **accuracy** and **macro-F1** — the main metric of the whole project, so that all models are compared the same way. Macro-F1 computes the F1 score for each class separately and averages them, so a model that ignores one class cannot get a good score.

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

### 6.4 Training settings

| Setting | Value | Why |
|---|---|---|
| learning rate | 2e-5 | Small steps, so the pre-trained knowledge is adapted, not destroyed |
| batch size | 16 (train) / 64 (eval) | ≈ 179 steps per epoch |
| epochs | 4 | Few: a small dataset overfits quickly |
| weight decay | 0.01 | Regularization against overfitting |
| evaluation / saving | every epoch | To follow validation scores and keep checkpoints |
| `load_best_model_at_end` | True, by macro-F1 | Keep the best epoch, not necessarily the last one |
| `fp16` | True | Mixed precision: faster on the GPU |
| seed | 42 | Reproducibility |

Checkpoints are saved in `models/`, which is excluded from Git (the files are too large for GitHub).

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert-irony",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    seed=42,
    report_to="none",
)

### 6.5 Training

The `Trainer` runs the full loop: for each batch, predict → compute the loss → adjust the weights; after each epoch, evaluate on validation and save a checkpoint. Training takes about 35 seconds on the GPU.

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.645253,0.613803,0.652356,0.646492
2,0.522039,0.628649,0.667016,0.666459
3,0.387162,0.647394,0.685864,0.685533
4,0.287587,0.707218,0.681675,0.681672


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=716, training_loss=0.46051007276140776, metrics={'train_runtime': 35.8977, 'train_samples_per_second': 318.572, 'train_steps_per_second': 19.946, 'total_flos': 115405235493564.0, 'train_loss': 0.46051007276140776, 'epoch': 4.0})

In [17]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.287587,0.647394,4,0.685864,0.685533


{'eval_loss': 0.6473937630653381,
 'eval_accuracy': 0.6858638743455497,
 'eval_f1_macro': 0.6855325202181186}

## 7. Results so far

**Validation results (DistilBERT, seed 42):**

| Epoch | Training loss | Validation loss | Accuracy | Macro-F1 |
|---|---|---|---|---|
| 1 | 0.645 | 0.614 | 0.652 | 0.646 |
| 2 | 0.522 | 0.629 | 0.667 | 0.666 |
| **3** | 0.387 | 0.647 | 0.686 | **0.686** |
| 4 | 0.288 | 0.707 | 0.682 | 0.682 |

**How to read it**
- The **training loss** decreases at every epoch: the model fits the training tweets better and better.
- The **validation loss** stops improving and increases from epoch 3: the model starts to **overfit** (it memorizes the training tweets and becomes overconfident on its mistakes). Macro-F1 stays around 0.68–0.69 while the loss rises, because F1 only counts right/wrong answers whereas the loss also measures confidence.
- Thanks to `load_best_model_at_end`, the model kept is the one with the best validation macro-F1 (epoch 3): `trainer.evaluate()` returns exactly epoch 3's numbers. (The displayed "Epoch 4" is where training stopped, not the epoch of the loaded weights.)

**Best validation macro-F1: 0.686.** Accuracy and macro-F1 are almost equal, which shows the model doesn't neglect one of the two classes.

**Run-to-run variability:** running the same code with the same seed three times gave 0.692, 0.688 and 0.686 (GPU computations in mixed precision are not perfectly deterministic). Differences below ~0.01 between two single runs are therefore not meaningful: the final comparison will use the **mean ± standard deviation over 3 seeds**.

For reference, the best system of the SemEval-2018 competition reached F1 = 0.71.

**Next steps**
1. Switch to a RoBERTa model pre-trained on tweets, which keeps capitals and emojis.
2. Train it with 3 different seeds and report mean ± std.
3. Final evaluation on the **test** set, together with the other models, and error analysis (tweets the transformer gets right and the classical models get wrong).

## 8. Twitter-RoBERTa

We now switch to a model better suited to our data: **Twitter-RoBERTa** (`cardiffnlp/twitter-roberta-base`), a RoBERTa-base model further pre-trained on about 60 million tweets by the Cardiff NLP team (the authors of TweetEval).

| | DistilBERT (uncased) | Twitter-RoBERTa |
|---|---|---|
| Size | ~67M parameters (distilled) | ~125M parameters |
| Capitals ("GREAT") | lost (lowercased) | kept |
| Emojis | `[UNK]` | real tokens |
| Pre-trained on | books + Wikipedia | RoBERTa data, then tweets |

All training settings are kept **identical** to the DistilBERT run, so that the only thing that changes is the model.

### 8.1 Matching the model's input format

During its pre-training on tweets, user mentions were replaced by `@user` and links by `http`. We apply the same replacement so our tweets look like what the model saw. This is not cleaning: capitals, punctuation and emojis are kept.

In [ ]:
def preprocess(text):
    words = []
    for word in text.split(" "):
        if word.startswith("@") and len(word) > 1:
            word = "@user"
        elif word.startswith("http"):
            word = "http"
        words.append(word)
    return " ".join(words)

print(preprocess("@SenRandPaul look at this http://t.co/Q2WB7riAvK lol"))

ds_tw = ds.map(lambda batch: {"text": [preprocess(t) for t in batch["text"]]}, batched=True)
print(ds_tw["train"][2]["text"])

### 8.2 Tokenizer of Twitter-RoBERTa

Each model needs its own tokenizer. We re-check the token lengths with this tokenizer to make sure `MAX_LENGTH = 64` still covers the tweets, then tokenize all splits.

In [ ]:
TW_MODEL_NAME = "cardiffnlp/twitter-roberta-base"
tw_tokenizer = AutoTokenizer.from_pretrained(TW_MODEL_NAME)

tw_lengths = pd.Series([len(tw_tokenizer(t)["input_ids"]) for t in ds_tw["train"]["text"]])
print("99th percentile:", tw_lengths.quantile(0.99), "| max:", tw_lengths.max())

def tw_tokenize(batch):
    return tw_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tw_tokenized_ds = ds_tw.map(tw_tokenize, batched=True)
print(tw_tokenized_ds)

### 8.3 Training

Same structure and settings as the DistilBERT training (section 6). Only the model, its tokenizer (and therefore the padding collator and the tokenized data) and the output folder change. The metric function is the same, so the scores are directly comparable.

In [ ]:
tw_model = AutoModelForSequenceClassification.from_pretrained(
    TW_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

tw_collator = DataCollatorWithPadding(tokenizer=tw_tokenizer)

tw_args = TrainingArguments(
    output_dir="../models/twitter-roberta-irony",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    seed=42,
    report_to="none",
)

tw_trainer = Trainer(
    model=tw_model,
    args=tw_args,
    train_dataset=tw_tokenized_ds["train"],
    eval_dataset=tw_tokenized_ds["validation"],
    data_collator=tw_collator,
    compute_metrics=compute_metrics,
)

tw_trainer.train()